# 02 - SQL analysis: Houston's economy in six questions

All queries live in `sql/*.sql` (SQLite dialect) and are re-run here against `db/lodes.db`. Each answers a question a workforce board or planning agency would ask.

| # | Question | SQL techniques |
|---|---|---|
| 1 | Is Houston growing as fast as Texas' other metros? | CTE, `LAG`, `FIRST_VALUE`, conditional-aggregation pivot, CAGR |
| 2 | What is the industry mix and how is it changing? | location quotient, `SUM() OVER`, HHI |
| 3 | Which counties are job hubs vs bedroom communities? | multi-CTE joins of WAC and RAC |
| 4 | Where are the job centers, and are jobs decentralising? | window `ROW_NUMBER` nearest-center assignment, CASE bands |
| 5 | How are commutes changing? | large OD fact table, weighted averages, denormalised keys |
| 6 | Where are low-wage workers stranded from jobs? | spatial self-join (bounding box + haversine UDF), `NTILE` quartiles |

In [1]:
import sys, sqlite3
from pathlib import Path
ROOT = Path.cwd() if (Path.cwd() / 'src').exists() else Path.cwd().parent
sys.path.insert(0, str(ROOT))
import pandas as pd, numpy as np
pd.set_option('display.width', 200); pd.set_option('display.max_columns', 30); pd.set_option('display.float_format', '{:,.2f}'.format)
from src.config import *
from src.run_sql import connect, parse_blocks
con = connect()
Q = {}
for f in sorted(SQL_DIR.glob('*.sql')):
    for name, sql in parse_blocks(f.read_text()):
        Q[name] = sql
def run(name, show_sql=True):
    if show_sql: print(Q[name], '\n')
    return pd.read_sql_query(Q[name], con)


## Q1. Growth vs. peer metros

In [2]:
run('metro_growth_summary')

WITH p AS (
  SELECT metro_name,
         MAX(CASE WHEN year = 2010 THEN jobs END) AS jobs_2010,
         MAX(CASE WHEN year = 2014 THEN jobs END) AS jobs_2014,
         MAX(CASE WHEN year = 2016 THEN jobs END) AS jobs_2016,
         MAX(CASE WHEN year = 2019 THEN jobs END) AS jobs_2019,
         MAX(CASE WHEN year = 2020 THEN jobs END) AS jobs_2020,
         MAX(CASE WHEN year = 2023 THEN jobs END) AS jobs_2023
  FROM v_metro_jobs GROUP BY metro_name)
SELECT metro_name, jobs_2010, jobs_2019, jobs_2023,
       ROUND(100 * (power(jobs_2019 * 1.0 / jobs_2010, 1.0 / 9) - 1), 2)  AS cagr_2010_19_pct,
       ROUND(100 * (power(jobs_2023 * 1.0 / jobs_2019, 1.0 / 4) - 1), 2)  AS cagr_2019_23_pct,
       ROUND(100.0 * (jobs_2016 - jobs_2014) / jobs_2014, 2)              AS chg_2014_16_pct,
       ROUND(100.0 * (jobs_2020 - jobs_2019) / jobs_2019, 2)              AS covid_chg_2019_20_pct,
       ROUND(100.0 * (jobs_2023 - jobs_2020) / jobs_2020, 2)              AS recovery_2020_23_pct,
       R

,metro_name,jobs_2010,jobs_2019,jobs_2023,cagr_2010_19_pct,cagr_2019_23_pct,chg_2014_16_pct,covid_chg_2019_20_pct,recovery_2020_23_pct,chg_2019_23_pct
0,Texas,10152390,12433128,13543073,2.28,2.16,4.05,-2.42,11.62,8.93
1,Dallas-Fort Worth,2870129,3719255,4175571,2.92,2.94,6.75,-1.89,14.44,12.27
2,Houston,2530059,3133166,3347878,2.40,1.67,3.38,-2.46,9.55,6.85
3,Austin,800498,1104004,1303090,3.64,4.23,10.63,-1.14,19.40,18.03
4,San Antonio,801317,1036482,1138312,2.90,2.37,6.81,-3.01,13.24,9.82


**Read:** Houston out-grew Texas before 2019 (2.40% vs 2.28%/yr) but lagged badly after (1.67% vs 2.16%). Dallas-Fort Worth (+12.3%) and Austin (+18.0%) pulled away in 2019-23 while Houston added +6.9%.

In [3]:
run('houston_shock_years', show_sql=False)

,year,jobs,mining_jobs,jobs_yoy_pct,mining_yoy_pct,mining_share_pct
0,2010,2530059,87676,NaN,NaN,3.47
1,2011,2636575,95781,4.21,9.24,3.63
2,2012,2680966,104423,1.68,9.02,3.89
3,2013,2780308,114741,3.71,9.88,4.13
4,2014,2889420,116427,3.92,1.47,4.03
5,2015,2986623,115435,3.36,-0.85,3.87
6,2016,2987075,100245,0.02,-13.16,3.36
7,2017,2990345,84899,0.11,-15.31,2.84
8,2018,3050225,86772,2.00,2.21,2.84
9,2019,3133166,91929,2.72,5.94,2.93


**Read:** the oil bust is visible - total job growth collapses to ~0% in 2016-17 as mining jobs fall 13-15% a year (Harvey hit in 2017 too). Mining employment then fell another 21% in 2021. Energy jobs move ~3x as violently as the whole economy.

## Q2. Industry mix

In [4]:
run('sector_change_houston_msa')

-- Job counts by industry in the Houston MSA; LQ = location quotient vs. Texas (>1 = specialised).
WITH msa AS (
  SELECT s.sector_code, s.year, SUM(s.jobs) AS jobs
  FROM fact_wac_sector s JOIN dim_tract d ON d.tract = s.tract WHERE d.in_houston_msa = 1 GROUP BY s.sector_code, s.year),
tx AS (SELECT sector_code, year, SUM(jobs) AS jobs FROM fact_wac_sector GROUP BY sector_code, year),
p AS (
  SELECT m.sector_code,
         MAX(CASE WHEN m.year = 2010 THEN m.jobs END) AS jobs_2010,
         MAX(CASE WHEN m.year = 2019 THEN m.jobs END) AS jobs_2019,
         MAX(CASE WHEN m.year = 2023 THEN m.jobs END) AS jobs_2023
  FROM msa m GROUP BY m.sector_code),
tot AS (
  SELECT (SELECT SUM(jobs) FROM msa WHERE year = 2023) AS msa_2023, (SELECT SUM(jobs) FROM tx WHERE year = 2023) AS tx_2023)
SELECT ds.sector_name, p.jobs_2010, p.jobs_2019, p.jobs_2023,
       p.jobs_2023 - p.jobs_2019                                                     AS abs_chg_2019_23,
       ROUND(100.0 * (p.jobs_2023 - p.

,sector_name,jobs_2010,jobs_2019,jobs_2023,abs_chg_2019_23,pct_chg_2019_23,pct_chg_2010_23,share_of_msa_pct,location_quotient
0,"Professional, Scientific & Technical",184489,241084,283326,42242,17.50,53.60,8.50,1.09
1,Health Care & Social Assistance,296140,385516,422334,36818,9.60,42.60,12.60,0.93
2,Transportation & Warehousing,109168,145831,173252,27421,18.80,58.70,5.20,1.09
3,Accommodation & Food Services,203323,283808,307331,23523,8.30,51.20,9.20,0.97
4,Educational Services,276934,304293,321526,17233,5.70,16.10,9.60,0.98
5,Retail Trade,266956,313591,329615,16024,5.10,23.50,9.90,0.95
6,"Admin, Support & Waste Mgmt",167680,220649,234671,14022,6.40,40.00,7.00,1.05
7,Finance & Insurance,89528,100701,114492,13791,13.70,27.90,3.40,0.74
8,Other Services,71157,90736,101761,11025,12.20,43.00,3.00,1.10
9,Wholesale Trade,139594,178095,185704,7609,4.30,33.00,5.60,1.14


**Read:** since 2019 Houston added most in Professional/Scientific/Technical (+42k), Health Care (+37k) and Transportation & Warehousing (+27k); Mining lost 17k. Mining is still the most over-represented sector (LQ 1.45); Finance (0.74) and Information (0.62) are under-represented.

In [5]:
run('sector_diversification_hhi', show_sql=False).query('year in [2010, 2015, 2019, 2023]').pivot(index='metro_name', columns='year', values='hhi')

year,2010,2015,2019,2023
metro_name,,,,
Austin,869.00,789.00,779.00,765.00
Dallas-Fort Worth,744.00,750.00,746.00,749.00
Houston,755.00,752.00,757.00,760.00
San Antonio,853.00,856.00,848.00,840.00


**Read:** the industry-concentration index for Houston (755 -> 760) is flat and equal to Dallas-Fort Worth (749) - so the energy decline has not left the region *more* concentrated. Austin and San Antonio are more concentrated (public sector and education).

## Q3. Hubs vs. bedroom communities

In [6]:
run('msa_county_summary', show_sql=False)

,county_name,jobs_2010,jobs_2019,jobs_2023,growth_2010_23_pct,growth_2019_23_pct,share_of_msa_jobs_pct,resident_workers_2023,jobs_per_resident_worker,high_wage_share_pct,low_wage_share_pct
0,Harris,2057904,2436871,2547424,23.80,4.50,76.10,2081817,1.22,61.40,14.30
1,Fort Bend,128331,213706,248090,93.30,16.10,7.40,412327,0.60,49.00,20.10
2,Montgomery,127475,196417,236728,85.70,20.50,7.10,314870,0.75,55.20,17.10
3,Galveston,91048,110891,122596,34.60,10.60,3.70,168228,0.73,51.90,19.80
4,Brazoria,80686,111134,117920,46.10,6.10,3.50,182047,0.65,55.20,17.90
5,Waller,10936,18470,22618,106.80,22.50,0.70,26810,0.84,58.80,15.90
6,Chambers,8907,17864,22262,149.90,24.60,0.70,25771,0.86,65.50,12.60
7,Liberty,13499,16877,18343,35.90,8.70,0.50,41767,0.44,51.20,17.00
8,Austin,10060,9083,9751,-3.10,7.40,0.30,16918,0.58,50.00,16.70


**Read:** Harris County (76% of MSA jobs) has 1.22 jobs per resident worker - a net importer. Fort Bend (0.60) and Montgomery (0.75) are net exporters, and they are also growing 3-4x faster than Harris (16% and 20% vs 4.5% since 2019).

## Q4. Employment centers and decentralisation

In [7]:
run('center_summary', show_sql=False)

,center,tracts,jobs_2010,jobs_2019,jobs_2023,share_of_msa_jobs_pct,growth_2019_23_pct,growth_2010_23_pct,high_wage_share_pct,low_wage_share_pct,health_care_share_pct,energy_mfg_share_pct
0,Rest of MSA,1238,1461090,1952607,2124149,63.50,8.80,45.40,55.60,16.70,10.30,11.10
1,Downtown,50,252269,256906,268996,8.00,4.70,6.60,71.80,10.90,5.20,6.10
2,Uptown / Galleria,65,243861,241981,247199,7.40,2.20,1.40,64.00,12.90,7.70,4.80
3,Texas Medical Center,49,161123,190208,203299,6.10,6.90,26.20,69.90,10.60,51.90,1.50
4,Westchase,64,96758,114725,116929,3.50,1.90,20.80,57.30,17.60,16.70,5.30
5,Energy Corridor,22,61535,74604,78309,2.30,5.00,27.30,74.80,9.80,7.00,13.00
6,Greenspoint / IAH,28,61304,66372,64912,1.90,-2.20,5.90,67.20,11.00,4.10,19.80
7,The Woodlands,13,37543,57071,64141,1.90,12.40,70.80,61.40,15.00,13.40,6.20
8,Sugar Land,15,38968,57142,60789,1.80,6.40,56.00,51.50,18.50,19.30,3.10
9,Clear Lake / NASA,21,54477,57670,58006,1.70,0.60,6.50,61.90,14.70,18.60,3.00


In [8]:
r = run('decentralization_by_ring', show_sql=False)
r[r.year.isin([2010, 2019, 2023])].pivot(index='ring', columns='year', values='share_pct')

year,2010,2019,2023
ring,,,
1. 0-5 mi,21.72,18.22,17.83
2. 5-10 mi,18.07,15.88,15.13
3. 10-20 mi,37.77,38.73,37.72
4. 20-35 mi,15.39,20.30,22.11
5. 35+ mi,7.05,6.87,7.21


**Read:** the 0-10 mile share of MSA jobs fell from 39.8% to 33.0% while the 20-35 mile ring rose from 15.4% to 22.1%. Downtown (+7% since 2010) and Uptown/Galleria (+1%) are nearly flat; The Woodlands (+71%), Sugar Land (+56%), Energy Corridor (+27%) and the Texas Medical Center (+26%) are growing. Baytown's industrial base shrank 15%.

## Q5. Commutes

In [9]:
run('commute_distance_trend')

-- Houston-MSA RESIDENTS: average one-way straight-line distance to the employer-assigned work location, and the
-- distance-band mix. CAVEAT: LODES places remote workers at their employer's address, so long-distance shares
-- overstate physical commuting after 2020; the mean is also pulled up by a long tail (bands are more robust).
SELECT o.year, SUM(o.jobs) AS workers,
       ROUND(SUM(o.dist_x_jobs) / SUM(o.jobs), 2)                    AS avg_miles_all,
       ROUND(SUM(o.dist_x_low)  / SUM(o.wage_low), 2)                AS avg_miles_low_wage,
       ROUND(SUM(o.dist_x_high) / SUM(o.wage_high), 2)               AS avg_miles_high_wage,
       ROUND(100.0 * SUM(o.jobs_lt10)   / SUM(o.jobs), 1)            AS pct_lt10mi,
       ROUND(100.0 * SUM(o.jobs_10_25)  / SUM(o.jobs), 1)            AS pct_10_25mi,
       ROUND(100.0 * SUM(o.jobs_25_50)  / SUM(o.jobs), 1)            AS pct_25_50mi,
       ROUND(100.0 * SUM(o.jobs_50plus) / SUM(o.jobs), 1)            AS pct_50plus_mi
FROM fact_od o

,year,workers,avg_miles_all,avg_miles_low_wage,avg_miles_high_wage,pct_lt10mi,pct_10_25mi,pct_25_50mi,pct_50plus_mi
0,2015,2853858,28.49,29.86,28.45,39.60,37.90,12.60,9.90
1,2019,2991753,30.80,32.01,31.27,37.80,37.90,13.40,10.90
2,2023,3232724,34.18,33.93,35.42,35.60,37.60,14.30,12.60


In [10]:
run('harris_inbound_outbound', show_sql=False)

,year,jobs_in_harris,filled_by_outside_residents,inbound_share_pct,harris_resident_workers,work_outside_harris,outbound_share_pct
0,2015,2346902,762138,32.50,1940847,356083,18.30
1,2019,2404677,826501,34.40,1984756,406580,20.50
2,2023,2517186,932732,37.10,2058191,473737,23.00


**Read:** the share of Houston-area workers with 25+ mile straight-line distances rose from 22.5% (2015) to 26.9% (2023). 37% of Harris jobs are now held by non-Harris residents (32.5% in 2015). *Caveat:* LODES puts remote workers at their employer's address, so some of the rise is not physical commuting; the direction, not the magnitude, is the finding.

In [11]:
run('home_county_commute_profile', show_sql=False).query('year == 2023')

,home_county,year,workers,avg_miles,pct_over_25mi,pct_work_in_harris
18,Harris,2023,2058191,31.40,20.40,77.00
19,Fort Bend,2023,407210,36.80,28.00,58.90
20,Montgomery,2023,311044,40.30,46.90,47.10
21,Brazoria,2023,180066,37.50,35.20,49.50
22,Galveston,2023,166322,36.90,34.70,43.10
23,Liberty,2023,41153,54.10,69.90,47.00
24,Waller,2023,26428,45.00,57.30,50.60
25,Chambers,2023,25491,46.30,53.70,55.70
26,Austin,2023,16819,50.90,63.30,31.70


## Q6. Spatial mismatch: who is far from low-wage jobs?

Three signals per tract (all 2023): (1) many low-wage resident workers, (2) few low-wage jobs per low-wage worker within 10 miles, (3) a high share of low-wage residents commuting 25+ miles. A tract scores 1 point per signal.

In [12]:
print(Q['setup_tract_pairs']); print(Q['setup_tract_access'])

-- All Houston-MSA tract pairs whose centroids are within 10 miles (bounding-box prefilter, then exact haversine).
DROP TABLE IF EXISTS tract_pairs;
CREATE TABLE tract_pairs AS
SELECT a.tract AS home_tract, b.tract AS job_tract, haversine_mi(a.lat, a.lon, b.lat, b.lon) AS dist_mi
FROM dim_tract a JOIN dim_tract b
  ON a.in_houston_msa = 1 AND b.in_houston_msa = 1 AND ABS(a.lat - b.lat) < 0.15 AND ABS(a.lon - b.lon) < 0.17
WHERE haversine_mi(a.lat, a.lon, b.lat, b.lon) <= 10;
CREATE INDEX ix_tp_h ON tract_pairs(home_tract);
CREATE INDEX ix_tp_j ON tract_pairs(job_tract);
DROP TABLE IF EXISTS tract_access;
CREATE TABLE tract_access AS
WITH jobs_side AS (
  SELECT p.home_tract AS tract, w.year, SUM(w.jobs) AS jobs_10mi, SUM(w.wage_low) AS low_wage_jobs_10mi
  FROM tract_pairs p JOIN fact_wac w ON w.tract = p.job_tract AND w.year IN (2019, 2023) GROUP BY p.home_tract, w.year),
workers_side AS (
  SELECT p.home_tract AS tract, r.year, SUM(r.wage_low) AS low_wage_workers_10mi
  FROM tract_pa

The `tract_pairs` table is a spatial self-join: every tract pair within 10 miles, found with a bounding-box prefilter (cheap, in C) then an exact haversine (a Python UDF registered on the SQLite connection) - ~1,600 x 1,600 candidate pairs in about 3 seconds, no GIS extension needed.

In [13]:
run('priority_summary_by_county', show_sql=False)

,county_name,tracts_screened,tracts_score3,tracts_score2plus,low_wage_residents_score3,pct_of_msa_low_wage_residents,pct_low_wage_25plus_mi_score3
0,Harris,1114,32,112,16728,3.31,37.70
1,Montgomery,133,16,71,8095,1.60,49.40
2,Fort Bend,133,12,38,6407,1.27,37.90
3,Galveston,101,9,32,5000,0.99,35.10
4,Brazoria,77,10,41,4935,0.98,47.40
5,Liberty,17,7,17,3894,0.77,67.20
6,Chambers,7,3,6,2433,0.48,55.30
7,Waller,11,2,10,1244,0.25,48.40
8,Austin,8,2,8,968,0.19,64.40


In [14]:
run('job_worker_ratio_by_county_2023', show_sql=False)

,county_name,low_wage_residents,lw_jobs_per_lw_worker_10mi,pct_lw_workers_25plus_mi
0,Harris,325701,1.15,21.70
1,Fort Bend,61731,0.92,26.60
2,Montgomery,46403,0.88,45.00
3,Galveston,28339,0.91,35.00
4,Brazoria,26373,0.70,37.90
5,Liberty,6537,0.48,69.60
6,Waller,4107,0.82,57.20
7,Chambers,3511,0.78,56.90
8,Austin,2780,0.59,64.50


**Read:** 93 tracts hit all three signals and hold ~49.7k low-wage workers (9.8% of the MSA's 505k). Brazoria (0.70) and Liberty (0.48) have far fewer low-wage jobs per low-wage worker within 10 miles than Harris (1.15); 70% of Liberty's low-wage residents commute 25+ miles versus 22% in Harris.

In [15]:
run('priority_tracts_top25', show_sql=False).head(10)

,tract,county_name,nearest_center,low_wage_residents,low_wage_jobs_10mi,lw_jobs_per_lw_worker_10mi,pct_lw_workers_25plus_mi,priority_score
0,48291700302,Liberty,Rest of MSA,985,2296,0.36,73.20,3
1,48201556000,Harris,Rest of MSA,924,5286,0.69,45.50,3
2,48071710201,Chambers,Rest of MSA,843,7095,0.74,55.40,3
3,48071710202,Chambers,Rest of MSA,813,6629,0.82,55.30,3
4,48473680100,Waller,Rest of MSA,778,25240,0.87,39.90,3
5,48071710100,Chambers,Rest of MSA,777,5837,0.86,55.30,3
6,48167723300,Galveston,Rest of MSA,744,15830,0.70,34.60,3
7,48339692402,Montgomery,Rest of MSA,712,22673,0.69,48.10,3
8,48339692802,Montgomery,Rest of MSA,693,6837,0.49,56.80,3
9,48039661902,Brazoria,Rest of MSA,681,3201,0.34,37.80,3
